# Benchmark LLMs vs regex V3 — Extraction d'articles

**Objectif.** Comparer plusieurs LLMs open-weights au regex V3 gelé sur la tâche d'extraction normalisée d'articles de codes juridiques français.

**Question de recherche.** À partir de quelle taille/archi un LLM bat-il le regex ? Symétriquement, quels LLMs rapides **ne parviennent pas à dépasser** une regex bien calibrée ?

## Ground truth

**20 arrêts annotés manuellement** (`../regex_v3/manual_annotations.json`) : 9 CC + 6 CA + 5 TJ, 177 pair_keys `<code_slug>:<article>`. Annotation : lecture intégrale de chaque arrêt, filtrage des codes officiels uniquement (exclut lois/décrets/conventions).

## Baseline regex V3

F1 = 0.920 (P=0.990, R=0.860). Détail dans `../docs_design/LLM-Shortlist-Extraction-Articles.md`.

## Modèles benchmarkés (7)

| Alias | HF ID | Params | Actifs | Licence |
|---|---|---|---|---|
| `gemma4-E2B` | `google/gemma-4-E2B-it` | ~2B eff. | 2B | Gemma |
| `qwen3.5-2B` | `Qwen/Qwen3.5-2B-Instruct` | 2B | 2B | Apache 2.0 |
| `gemma4-E4B` | `google/gemma-4-E4B-it` | ~4B eff. | 4B | Gemma |
| `ministral-8B` | `mistralai/Ministral-8B-Instruct-2410` | 8B | 8B | Mistral-Research |
| `qwen3.5-9B` | `Qwen/Qwen3.5-9B-Instruct` | 9B | 9B | Apache 2.0 |
| `gemma4-26B-A4B` (MoE) | `google/gemma-4-26B-A4B-it` | 26B | 4B | Gemma |
| `gemma4-31B` | `google/gemma-4-31B-it` | 31B | 31B | Gemma |

## Protocole

1. Setup cluster (identique à `validate_article_regex_1.ipynb`).
2. **Pour chaque modèle**, au choix dans la cellule CONFIG : download → serve → extraction sur 20 arrêts → kill serveur → sauvegarde JSON.
3. Cellule finale : chargement de tous les résultats + regex V3 → tableau comparatif P/R/F1/latence/tokens.

Le **prompt et le post-processing sont identiques pour tous les LLMs** (module `llm_extract_articles.py`), garantissant que les différences observées viennent du modèle seul.

---
## 0. Setup cluster — install, download, démarrage vLLM

Cellules reprises de `validate_article_regex_1.ipynb`. À ré-exécuter à chaque changement de modèle (0.2 pour changer MODEL_ID, puis 0.4 → 0.6).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.1 — REGISTRE DES MODÈLES À BENCHMARKER
# IDs HuggingFace vérifiés (avril 2026).
# ═══════════════════════════════════════════════════════════════════════

MODEL_REGISTRY = {
    # alias court : (hf_id, note)
    "gemma4-E2B":       ("google/gemma-4-E2B-it",                  "Gemma 4 E2B-IT (effectif ~2B, MatFormer)"),
    "qwen3.5-2B":       ("Qwen/Qwen3.5-2B-Instruct",                "Qwen3.5 dense 2B, 256k ctx"),
    "gemma4-E4B":       ("google/gemma-4-E4B-it",                  "Gemma 4 E4B-IT (effectif ~4B)"),
    "ministral-8B":     ("mistralai/Ministral-8B-Instruct-2410",    "Ministral 8B — FR natif Mistral"),
    "qwen3.5-9B":       ("Qwen/Qwen3.5-9B-Instruct",                "Qwen3.5 dense 9B, long ctx"),
    "gemma4-26B-A4B":   ("google/gemma-4-26B-A4B-it",               "Gemma 4 MoE 26B / 4B actifs"),
    "gemma4-31B":       ("google/gemma-4-31B-it",                  "Gemma 4 dense 31B (= baseline notebook précédent)"),
}

for alias, (hf, note) in MODEL_REGISTRY.items():
    print(f"  {alias:<18} → {hf:<45} | {note}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.2 — CONFIG CLUSTER — ⚠ CHANGER MODEL_ALIAS ENTRE DEUX RUNS
# ═══════════════════════════════════════════════════════════════════════
import os, subprocess
from pathlib import Path

# ── CHOIX DU MODÈLE ──────────────────────────────────────────────────
MODEL_ALIAS = "ministral-8B"    # ← EDIT ME : alias du registre ci-dessus
MODEL_ID    = MODEL_REGISTRY[MODEL_ALIAS][0]

VLLM_PORT = 8000
# Paramètres VRAM — à ajuster selon la taille du modèle (L40S 40 Go)
MAX_LEN   = 32768
GPU_UTIL  = 0.90

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    )
    gpus = [l for l in out.strip().split("\n") if l]
    NUM_GPUS = len(gpus)
    print(f"GPUs détectés : {NUM_GPUS}")
    for i, g in enumerate(gpus):
        print(f"  [{i}] {g}")
except Exception as e:
    print(f"[WARN] nvidia-smi indisponible ({e}) → NUM_GPUS=1")
    NUM_GPUS = 1

LOG_DIR = Path("./logs"); LOG_DIR.mkdir(exist_ok=True)
VLLM_LOG = LOG_DIR / f"vllm_{MODEL_ALIAS}.log"
VLLM_PID = LOG_DIR / "vllm.pid"

VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}/v1"
print(f"\nAlias         : {MODEL_ALIAS}")
print(f"HF ID         : {MODEL_ID}")
print(f"Contexte max  : {MAX_LEN} tokens")
print(f"vLLM serveur  : {VLLM_BASE_URL}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.3 — INSTALLATION DES DÉPENDANCES (identique à validate_article_regex_1)
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, importlib

def _ensure_pip():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return
    except (subprocess.CalledProcessError, FileNotFoundError):
        pass
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"])

_ensure_pip()

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                       "--force-reinstall", "--no-deps",
                       "numpy>=1.26,<2", "scipy>=1.11,<1.14"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                       "--force-reinstall",
                       "cffi>=1.17", "cryptography>=42", "pyOpenSSL>=24",
                       "urllib3>=1.26,<3", "boto3>=1.34", "botocore>=1.34"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "vllm>=0.8.5", "openai>=1.50", "pydantic>=2",
                       "pandas>=2", "pyarrow>=14", "huggingface_hub>=0.25",
                       "tqdm>=4.65", "rich>=13", "tabulate>=0.9"])
importlib.invalidate_caches()
print("✓ Dépendances installées.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.4 — AUTH HUGGINGFACE + DOWNLOAD
# ═══════════════════════════════════════════════════════════════════════
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✓ Authentifié sur HuggingFace")
else:
    print("[WARN] HF_TOKEN non défini (export HF_TOKEN=hf_xxx avant de lancer Jupyter)")

print(f"\nTéléchargement de {MODEL_ID}…")
model_path = snapshot_download(repo_id=MODEL_ID, ignore_patterns=["*.md", "*.txt", "original/*"])
print(f"\n✓ Modèle en cache : {model_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.5 — DÉMARRAGE SERVEUR vLLM
# ═══════════════════════════════════════════════════════════════════════
import subprocess, os, signal, sys, time

if VLLM_PID.exists():
    try:
        old = int(VLLM_PID.read_text())
        os.killpg(os.getpgid(old), signal.SIGTERM)
        print(f"Serveur précédent (PID={old}) arrêté")
        time.sleep(3)
    except (ProcessLookupError, ValueError, PermissionError):
        pass

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--tensor-parallel-size", str(NUM_GPUS),
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--port", str(VLLM_PORT),
]
print("Commande :", " ".join(cmd))

log_f = open(VLLM_LOG, "w")
vllm_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid)
VLLM_PID.write_text(str(vllm_proc.pid))
print(f"✓ vLLM démarré (PID={vllm_proc.pid})  ·  logs → {VLLM_LOG}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.6 — ATTENTE DU DÉMARRAGE
# ═══════════════════════════════════════════════════════════════════════
import time, urllib.request, json

HEALTH_URL = f"http://localhost:{VLLM_PORT}/health"
MAX_WAIT_S = 900

t0 = time.time()
ready = False
while time.time() - t0 < MAX_WAIT_S:
    if vllm_proc.poll() is not None:
        raise RuntimeError(f"vLLM s'est arrêté (code={vllm_proc.returncode}) — inspecter {VLLM_LOG}")
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=2) as r:
            if r.status == 200:
                ready = True; break
    except Exception:
        pass
    print(f"  chargement… {int(time.time()-t0)}s (max {MAX_WAIT_S}s)", end="\r")
    time.sleep(5)

if not ready:
    raise RuntimeError(f"vLLM non prêt après {MAX_WAIT_S}s — voir {VLLM_LOG}")

print(f"\n✓ vLLM prêt  (démarrage : {int(time.time()-t0)}s)")
with urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models") as r:
    data = json.load(r)
print("Modèles servis :", [m['id'] for m in data.get('data', [])])

---
## 1. Configuration benchmark


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CONFIG EXTRACTION
# ═══════════════════════════════════════════════════════════════════════
API_KEY         = "EMPTY"
TEMPERATURE     = 0.0
MAX_TOKENS_OUT  = 2048     # listings longs possibles (CESEDA: 15 pair_keys)

# ─── Résolution des chemins : local d'abord, puis parent ─────────────
HERE = Path(".").resolve()

def _resolve(*candidates: Path) -> Path:
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"Aucun des chemins n'existe : {candidates}")

SAMPLE_PATH = _resolve(
    HERE / "cluster_data" / "regex_validation" / "sample_100.jsonl",
    HERE.parent / "cluster_data" / "regex_validation" / "sample_100.jsonl",
)
ANNOTATIONS = _resolve(
    HERE / "manual_annotations.json",
    HERE.parent / "regex_v3" / "manual_annotations.json",
)
RESULTS_DIR = Path("./results"); RESULTS_DIR.mkdir(exist_ok=True)

print(f"Sample  : {SAMPLE_PATH}")
print(f"Ann.    : {ANNOTATIONS}")
print(f"Out     : {RESULTS_DIR.resolve()}")


---
## 2. Imports + chargement GT + baseline regex V3


In [ ]:
import json, time, sys
from openai import OpenAI
from tqdm import tqdm

# Les modules regex + LLM sont à côté du notebook (autonome sur cluster)
sys.path.insert(0, str(HERE))

from iterate_regex        import extract_pairs_v3
from llm_extract_articles import (
    extract_pairs_llm, LLM_SYSTEM_PROMPT, CODE_SLUGS,
)

print(f"Prompt normé : {len(LLM_SYSTEM_PROMPT)} chars · {len(CODE_SLUGS)} codes autorisés")


In [ ]:
# ─── Chargement des 20 arrêts annotés + ground truth ────────────────
def load_sample_local(path: Path) -> dict[str, dict]:
    """Charge le JSONL pré-échantillonné (100 arrêts)."""
    out = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            rec = json.loads(line)
            out[rec["id"]] = rec
    return out

recs = load_sample_local(SAMPLE_PATH)
ann  = json.loads(ANNOTATIONS.read_text())["annotations"]

assert len(ann) == 20, f"Attendu 20 annotations, trouvé {len(ann)}"
from collections import Counter
print(f"Arrêts annotés : {len(ann)}  " + str(Counter(v['jur'] for v in ann.values())))

# Vérifie que tous les arrêts annotés sont dans le sample
missing = [rid for rid in ann if rid not in recs]
assert not missing, f"Arrêts manquants dans le sample : {missing}"

gt_pairs = {rid: set(v['gt']) for rid, v in ann.items()}
total_gt = sum(len(s) for s in gt_pairs.values())
print(f"Total pair_keys GT : {total_gt}")


In [ ]:
# ─── Baseline regex V3 (gelée) ───────────────────────────────────────
def prf(tp, fp, fn):
    p  = tp/(tp+fp) if tp+fp else 0.0
    r  = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f1

regex_results = {"per_arret": [], "totals": {"tp": 0, "fp": 0, "fn": 0}}
for rid, gt in gt_pairs.items():
    pred = extract_pairs_v3(recs[rid]["text"])
    tp, fp, fn = gt & pred, pred - gt, gt - pred
    regex_results["totals"]["tp"] += len(tp)
    regex_results["totals"]["fp"] += len(fp)
    regex_results["totals"]["fn"] += len(fn)
    regex_results["per_arret"].append({
        "id": rid, "jur": ann[rid]["jur"], "n_gt": len(gt), "n_pred": len(pred),
        "tp": len(tp), "fp": len(fp), "fn": len(fn),
        "missed": sorted(fn), "extra": sorted(fp),
    })
t = regex_results["totals"]
p, r, f1 = prf(t["tp"], t["fp"], t["fn"])
regex_results["metrics"] = {"precision": p, "recall": r, "f1": f1}
print(f"Regex V3 — TP={t['tp']}  FP={t['fp']}  FN={t['fn']}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")

(RESULTS_DIR / "regex-v3.json").write_text(
    json.dumps({"alias": "regex-v3", **regex_results}, ensure_ascii=False, indent=2))

---
## 3. Vérification serveur LLM courant


In [ ]:
client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)
try:
    models = client.models.list()
    print("Serveur OK — modèles :", [m.id for m in models.data])
    assert any(MODEL_ID in m.id or m.id in MODEL_ID for m in models.data), \
        f"MODEL_ID={MODEL_ID} absent des modèles servis"
except Exception as e:
    raise RuntimeError(f"Serveur vLLM indisponible : {e}")

---
## 4. Extraction LLM sur les 20 arrêts annotés

Prompt et schéma JSON strict sont définis dans `llm_extract_articles.py` (importé ci-dessus). La même version est utilisée pour **tous les LLMs** → les différences observées viennent du modèle.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EXTRACTION LLM sur les 20 arrêts annotés
# ═══════════════════════════════════════════════════════════════════════
per_arret = []
latencies, tokens_counts = [], []
totals = {"tp": 0, "fp": 0, "fn": 0}
slug_known = set(CODE_SLUGS.keys())

for rid, gt in tqdm(list(gt_pairs.items()), desc=f"LLM {MODEL_ALIAS}"):
    text = recs[rid]["text"]
    pred, meta = extract_pairs_llm(text, client, MODEL_ID, max_tokens=MAX_TOKENS_OUT)

    tp, fp, fn = gt & pred, pred - gt, gt - pred
    totals["tp"] += len(tp); totals["fp"] += len(fp); totals["fn"] += len(fn)

    if meta.get("latency_s") is not None:
        latencies.append(meta["latency_s"])
    if meta.get("tokens_used"):
        tokens_counts.append(meta["tokens_used"])

    # Catégorisation FP
    fp_bad_slug = sum(1 for pk in fp if pk.split(":", 1)[0] not in slug_known)
    fp_hallu    = len(fp) - fp_bad_slug

    per_arret.append({
        "id": rid, "jur": ann[rid]["jur"],
        "n_gt": len(gt), "n_pred": len(pred),
        "tp": len(tp), "fp": len(fp), "fn": len(fn),
        "missed": sorted(fn), "extra": sorted(fp),
        "fp_bad_slug": fp_bad_slug, "fp_hallucination": fp_hallu,
        "latency_s": meta.get("latency_s"),
        "tokens_used": meta.get("tokens_used"),
        "finish_reason": meta.get("finish_reason"),
        "error": meta.get("error"),
    })

p, r, f1 = prf(totals["tp"], totals["fp"], totals["fn"])
print(f"\n{MODEL_ALIAS}")
print(f"  TP={totals['tp']}  FP={totals['fp']}  FN={totals['fn']}")
print(f"  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")
if latencies:
    print(f"  Latence  moy={sum(latencies)/len(latencies):.2f}s  "
          f"p50={sorted(latencies)[len(latencies)//2]:.2f}s")
if tokens_counts:
    print(f"  Tokens   moy={sum(tokens_counts)/len(tokens_counts):.0f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# SAUVEGARDE du run courant
# ═══════════════════════════════════════════════════════════════════════
run_report = {
    "alias": MODEL_ALIAS,
    "model_id": MODEL_ID,
    "metrics": {"precision": p, "recall": r, "f1": f1},
    "totals": totals,
    "latency_mean_s": sum(latencies)/len(latencies) if latencies else None,
    "latency_p50_s":  sorted(latencies)[len(latencies)//2] if latencies else None,
    "tokens_mean":    sum(tokens_counts)/len(tokens_counts) if tokens_counts else None,
    "per_arret": per_arret,
}
out_path = RESULTS_DIR / f"{MODEL_ALIAS}.json"
out_path.write_text(json.dumps(run_report, ensure_ascii=False, indent=2))
print(f"✓ Résultats sauvés : {out_path}")

---
## 5. Arrêt du serveur vLLM

À exécuter avant de changer de modèle (retour à la cellule 0.2 avec nouveau `MODEL_ALIAS`).

In [ ]:
import os, signal, time
if VLLM_PID.exists():
    try:
        pid = int(VLLM_PID.read_text())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print(f"Serveur (PID={pid}) arrêté")
        time.sleep(3)
        VLLM_PID.unlink(missing_ok=True)
    except (ProcessLookupError, ValueError, PermissionError) as e:
        print(f"[WARN] Pas de serveur à arrêter : {e}")
else:
    print("Aucun serveur en cours.")

---
## 5bis. 🚀 Run full — boucle sur tous les modèles du registre

Lance séquentiellement les 7 modèles : download → serve → extract → kill → ligne CSV. Skip auto si `results/<alias>.json` existe déjà. Reprenable sur interruption.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RUN FULL — boucle sur tous les modèles du MODEL_REGISTRY
# Pour chaque : kill server → download → start → wait → extract → kill
# CSV : une ligne par modèle (alias, F1, P, R, latence, tokens)
# ═══════════════════════════════════════════════════════════════════════
import os, signal, subprocess, sys, time, urllib.request, json, csv
from huggingface_hub import snapshot_download
from openai import OpenAI
from tqdm import tqdm

CSV_PATH = RESULTS_DIR / "comparison.csv"
CSV_HEADER = ["alias", "model_id", "TP", "FP", "FN",
              "precision", "recall", "f1",
              "latency_mean_s", "tokens_mean", "status"]

# Initialise CSV si absent + baseline regex V3 en première ligne
if not CSV_PATH.exists():
    with open(CSV_PATH, "w", newline="") as f:
        w = csv.writer(f); w.writerow(CSV_HEADER)
        m = regex_results["metrics"]; t = regex_results["totals"]
        w.writerow(["regex-v3", "regex-v3-frozen", t["tp"], t["fp"], t["fn"],
                    round(m["precision"],3), round(m["recall"],3), round(m["f1"],3),
                    "", "", "ok"])

def kill_vllm():
    if VLLM_PID.exists():
        try:
            pid = int(VLLM_PID.read_text())
            os.killpg(os.getpgid(pid), signal.SIGTERM)
            time.sleep(5)
        except (ProcessLookupError, ValueError, PermissionError):
            pass
        VLLM_PID.unlink(missing_ok=True)

def start_vllm(model_id: str, log_path: Path) -> subprocess.Popen:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_id,
           "--tensor-parallel-size", str(NUM_GPUS),
           "--max-model-len", str(MAX_LEN),
           "--gpu-memory-utilization", str(GPU_UTIL),
           "--port", str(VLLM_PORT)]
    log_f = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid)
    VLLM_PID.write_text(str(proc.pid))
    return proc

def wait_vllm(proc, timeout_s: int = 900) -> bool:
    health = f"http://localhost:{VLLM_PORT}/health"
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return False
        try:
            with urllib.request.urlopen(health, timeout=2) as r:
                if r.status == 200: return True
        except Exception:
            pass
        time.sleep(5)
    return False

def append_csv(row: dict):
    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerow([row.get(k, "") for k in CSV_HEADER])

# ─── BOUCLE PRINCIPALE ─────────────────────────────────────────────
for alias, (hf_id, note) in MODEL_REGISTRY.items():
    out_json = RESULTS_DIR / f"{alias}.json"
    if out_json.exists():
        print(f"[{alias}] déjà fait → skip (supprime {out_json} pour re-run)")
        continue

    print(f"\n{'='*70}\n[{alias}] {hf_id}\n{'='*70}")
    try:
        kill_vllm()
        print("  ↳ download…"); t0 = time.time()
        snapshot_download(repo_id=hf_id, ignore_patterns=["*.md","*.txt","original/*"])
        print(f"  ↳ download OK ({int(time.time()-t0)}s)")

        print("  ↳ start vLLM…"); t0 = time.time()
        log_path = LOG_DIR / f"vllm_{alias}.log"
        proc = start_vllm(hf_id, log_path)
        if not wait_vllm(proc):
            raise RuntimeError(f"vLLM KO — voir {log_path}")
        print(f"  ↳ vLLM prêt ({int(time.time()-t0)}s)")

        client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)
        per_arret, latencies, tokens_counts = [], [], []
        totals = {"tp":0,"fp":0,"fn":0}
        slug_known = set(CODE_SLUGS.keys())

        for rid, gt in tqdm(list(gt_pairs.items()), desc=alias):
            pred, meta = extract_pairs_llm(recs[rid]["text"], client, hf_id,
                                           max_tokens=MAX_TOKENS_OUT)
            tp, fp, fn = gt & pred, pred - gt, gt - pred
            totals["tp"]+=len(tp); totals["fp"]+=len(fp); totals["fn"]+=len(fn)
            if meta.get("latency_s") is not None: latencies.append(meta["latency_s"])
            if meta.get("tokens_used"): tokens_counts.append(meta["tokens_used"])
            fp_bad = sum(1 for pk in fp if pk.split(":",1)[0] not in slug_known)
            per_arret.append({
                "id": rid, "jur": ann[rid]["jur"],
                "n_gt": len(gt), "n_pred": len(pred),
                "tp": len(tp), "fp": len(fp), "fn": len(fn),
                "missed": sorted(fn), "extra": sorted(fp),
                "fp_bad_slug": fp_bad, "fp_hallucination": len(fp)-fp_bad,
                "latency_s": meta.get("latency_s"),
                "tokens_used": meta.get("tokens_used"),
                "finish_reason": meta.get("finish_reason"),
                "error": meta.get("error"),
            })

        p = totals["tp"]/(totals["tp"]+totals["fp"]) if totals["tp"]+totals["fp"] else 0.0
        r = totals["tp"]/(totals["tp"]+totals["fn"]) if totals["tp"]+totals["fn"] else 0.0
        f1 = 2*p*r/(p+r) if p+r else 0.0
        lat_mean = sum(latencies)/len(latencies) if latencies else None
        tok_mean = sum(tokens_counts)/len(tokens_counts) if tokens_counts else None

        report = {"alias": alias, "model_id": hf_id,
                  "metrics": {"precision": p, "recall": r, "f1": f1},
                  "totals": totals,
                  "latency_mean_s": lat_mean, "tokens_mean": tok_mean,
                  "per_arret": per_arret}
        out_json.write_text(json.dumps(report, ensure_ascii=False, indent=2))
        append_csv({
            "alias": alias, "model_id": hf_id,
            "TP": totals["tp"], "FP": totals["fp"], "FN": totals["fn"],
            "precision": round(p,3), "recall": round(r,3), "f1": round(f1,3),
            "latency_mean_s": round(lat_mean,2) if lat_mean else "",
            "tokens_mean": int(tok_mean) if tok_mean else "",
            "status": "ok",
        })
        print(f"  ✓ [{alias}] P={p:.3f}  R={r:.3f}  F1={f1:.3f}  "
              f"lat={lat_mean:.2f}s" if lat_mean else f"  ✓ F1={f1:.3f}")

    except Exception as e:
        print(f"  ✗ [{alias}] ERREUR : {e}")
        append_csv({"alias": alias, "model_id": hf_id, "status": f"error: {e}"})
    finally:
        kill_vllm()

print(f"\n✓ Terminé — CSV : {CSV_PATH}")
import pandas as pd
display(pd.read_csv(CSV_PATH))


---
## 6. Tableau comparatif final

À exécuter après avoir run **tous** les modèles souhaités (fichiers `results/<alias>.json` tous générés).

In [ ]:
import pandas as pd

rows = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    rep = json.loads(f.read_text())
    rows.append({
        "alias":     rep["alias"],
        "TP":        rep["totals"]["tp"],
        "FP":        rep["totals"]["fp"],
        "FN":        rep["totals"]["fn"],
        "P":         round(rep["metrics"]["precision"], 3),
        "R":         round(rep["metrics"]["recall"], 3),
        "F1":        round(rep["metrics"]["f1"], 3),
        "lat_s":     round(rep["latency_mean_s"], 2) if rep.get("latency_mean_s") else None,
        "tokens":    int(rep["tokens_mean"]) if rep.get("tokens_mean") else None,
    })

df = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(df)

# Sauvegarde tableau
df.to_csv(RESULTS_DIR / "comparison.csv", index=False)
print(f"\n✓ Tableau sauvé : {RESULTS_DIR / 'comparison.csv'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DÉTAIL PAR ARRÊT — pour diagnostiquer où les LLMs échouent
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd

detail_rows = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    rep = json.loads(f.read_text())
    for rec in rep["per_arret"]:
        p_, r_, f1_ = prf(rec["tp"], rec["fp"], rec["fn"])
        detail_rows.append({
            "alias": rep["alias"], "id": rec["id"], "jur": rec["jur"],
            "n_gt": rec["n_gt"], "n_pred": rec["n_pred"],
            "TP": rec["tp"], "FP": rec["fp"], "FN": rec["fn"], "F1": round(f1_, 3),
        })

df_detail = pd.DataFrame(detail_rows)
# Pivot F1 par arrêt × modèle
pivot = df_detail.pivot(index=["id", "jur"], columns="alias", values="F1")
display(pivot)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC : TYPOLOGIE DES ERREURS LLM
# FP_bad_slug = LLM a inventé un slug hors liste → erreur de forme
# FP_hallu    = LLM a inventé un article dans un code valide → hallu
# FN          = LLM a omis un article du GT
# ═══════════════════════════════════════════════════════════════════════
typo_rows = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    rep = json.loads(f.read_text())
    if rep["alias"] == "regex-v3":
        continue
    bs = sum(r.get("fp_bad_slug", 0) for r in rep["per_arret"])
    hl = sum(r.get("fp_hallucination", 0) for r in rep["per_arret"])
    fn = sum(r["fn"] for r in rep["per_arret"])
    typo_rows.append({"alias": rep["alias"],
                      "FP_bad_slug": bs, "FP_hallucination": hl, "FN_omission": fn})

df_typo = pd.DataFrame(typo_rows).sort_values("alias")
display(df_typo)